In [19]:
# Crafting an AI-Powered HR Assistant: A Use Case for Nestle’s HR Policy Documents

In [20]:
# Overview
# The project aims to create a conversational chatbot that responds to user inquiries using PDF document information.
#  It requires proficiency in extracting and converting text into numerical vectors, establishing an answer-finding mechanism, and designing a user-friendly chatbot interface with Gradio.
# Additionally, the initiative emphasizes structuring inquiries for clear communication and deploying the chatbot for practical use, guaranteeing the system's accessibility and efficiency in meeting user needs.

In [21]:
!pip install -U \
langchain==0.3.27 \
langchain-core==0.3.79 \
langchain-community==0.3.31 \
langchain-openai==0.3.35 \
langchain-text-splitters==0.3.9 \
langchainhub \
faiss-cpu \
sentence-transformers -q


In [22]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.chains import VectorDBQA, RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader, PyPDFLoader
import openai
import os

In [23]:
!pip install pypdf -q

In [33]:
loader = PyPDFLoader('/content/the_nestle_hr_policy_pdf_2012.pdf')
documents = loader.load()

In [35]:
from google.colab import userdata
x = userdata.get('API_KEYS')

In [36]:
# The API key 'MYAPIKEY' stored in Colab secrets needs to be verified.
# The `AuthenticationError` indicates an issue with the OpenAI API key used for embeddings.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

In [37]:
texts = text_splitter.split_documents(documents)

In [38]:
embeddings = OpenAIEmbeddings(api_key=x)

In [39]:
vectordb = FAISS.from_documents(texts, embeddings)

In [40]:
from langchain_openai import ChatOpenAI
qa = RetrievalQA.from_chain_type(llm=ChatOpenAI(model_name="gpt-3.5-turbo", api_key=x), chain_type="stuff", retriever=vectordb.as_retriever())

In [41]:
from langchain import PromptTemplate
template = """
I am an HR Assistant turning people processes into positive experiences.
Please answer the following questions in clear and professional English.

Question : {question}
Answer :
"""

prompt = PromptTemplate(input_variables=["question"],template=template,)

In [42]:
# 1. Import Gradio library
import gradio as gr

# Define the actual QA system using the qa chain
def get_nestle_answer(user_question, history):
    # Ensure history is initialized as a list of dictionaries if it's None or empty
    history = history or []

    # Using the brief prompt template as requested by the user
    brief_prompt_template = "answer this question briefly: {question}"
    formatted_question = brief_prompt_template.format(question=user_question)

    # Get the answer from the QA chain
    try:
        answer = qa.run(formatted_question)
    except Exception as e:
        answer = f"Error retrieving answer: {e}"

    # Append messages in the required dictionary format for type='messages'
    history.append({"role": "user", "content": user_question})
    history.append({"role": "assistant", "content": answer})

    return history, ""

# 5. Build Gradio interface using gr.Blocks for chat history
with gr.Blocks(title="Nestlé HR Policy Chatbot", theme='soft', css="body { background-color: #f5f5dc; }") as demo:
    gr.Markdown("## 📝☕ Nestlé HR Policy Chatbot")
    gr.Markdown("Your AI-powered HR Assistant for Nestlé HR policies. Ask me anything about the company's HR documents!")

    chatbot = gr.Chatbot(label="Chat History", height=400, type='messages', allow_tags=False)
    user_input = gr.Textbox(placeholder="Ask a question about Nestlé's HR Policy...", label="Your Question")
    send_btn = gr.Button("Send")
    clear_btn = gr.Button("Clear Chat")

    send_btn.click(
        get_nestle_answer,
        inputs=[user_input, chatbot],
        outputs=[chatbot, user_input]
    )
    user_input.submit(
        get_nestle_answer,
        inputs=[user_input, chatbot],
        outputs=[chatbot, user_input]
    )
    clear_btn.click(lambda: ([], ""), inputs=None, outputs=[chatbot, user_input])

# 6. Launch the app
demo.launch(share=True)

/tmp/ipykernel_2068/2364487903.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Nestlé HR Policy Chatbot", theme='soft', css="body { background-color: #f5f5dc; }") as demo:
/tmp/ipykernel_2068/2364487903.py:26: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Nestlé HR Policy Chatbot", theme='soft', css="body { background-color: #f5f5dc; }") as demo:
/tmp/ipykernel_2068/2364487903.py:30: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=400, type='messages', allow_tags=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c84b7ec0917a51f4aa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
